# Step 1: CTA Ridership Data Cleaning & Validation

## Objectives
1. Load the raw CTA Bus Daily Totals dataset[cite: 1, 2].
2. Parse dates explicitly to `YYYY-MM-DD` and clean numerical fields (removing commas).
3. Validate and drop duplicate route/date records.
4. Extract the primary 2-Year recent demand modeling window (`2024-08-01` to `2026-07-31`)[cite: 1].
5. Export clean datasets to Parquet format for fast loading in subsequent steps[cite: 1].

In [1]:
import os
import pandas as pd

# Define relative paths based on project structure
RAW_RIDERSHIP_PATH = os.path.join("..", "data", "raw", "ridership")
PROCESSED_DATA_PATH = os.path.join("..", "data", "processed")

# Ensure the output directory exists
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

print("Paths configured successfully.")

Paths configured successfully.


## Load Raw Data and Clean Fields
We locate the raw ridership CSV, standardize column names, parse dates, and remove comma formatting from the `rides` column[cite: 1].

In [2]:
# Locate the CSV file dynamically in the ridership folder
csv_files = [f for f in os.listdir(RAW_RIDERSHIP_PATH) if f.endswith('.csv')]
if not csv_files:
    raise FileNotFoundError(f"No CSV file found in {RAW_RIDERSHIP_PATH}")

csv_path = os.path.join(RAW_RIDERSHIP_PATH, csv_files[0])
print(f"Loading raw ridership file: {csv_path}")

# Load raw dataset
df = pd.read_csv(csv_path)
print(f"Raw record count: {len(df):,}")

# Standardize column names (strip whitespace and convert to lowercase)
df.columns = df.columns.str.strip().str.lower()

# Format route string and parse date
df['route'] = df['route'].astype(str).str.strip()
df['date'] = pd.to_datetime(df['date'], format='%m/%d/%Y', errors='coerce')

# Remove comma separators from 'rides' and convert to integer
if df['rides'].dtype == 'object':
    df['rides'] = df['rides'].str.replace(',', '')
df['rides'] = pd.to_numeric(df['rides'], errors='coerce').fillna(0).astype(int)

print("Data parsing complete. Displaying head:")
df.head()

Loading raw ridership file: ..\data\raw\ridership\CTA_-_Ridership_-_Bus_Routes_-_Daily_Totals_by_Route.csv
Raw record count: 1,120,604
Data parsing complete. Displaying head:


,route,date,daytype,rides
0,1,2001-05-14,W,6217
1,1,2001-07-06,W,6590
2,1,2001-01-04,W,6907
3,1,2001-04-19,W,6568
4,1,2001-10-04,W,6361


## Deduplication & Quality Audit
Validate uniqueness across `(route, date)` pairs and check summary statistics[cite: 1].

In [3]:
# Check for and remove duplicate route/date observations
initial_count = len(df)
df = df.drop_duplicates(subset=['route', 'date'])
dropped_duplicates = initial_count - len(df)

print(f"Initial rows: {initial_count:,}")
print(f"Duplicates removed: {dropped_duplicates:,}")
print(f"Clean unique observations: {len(df):,}")

# Verify true zero-count records (CTA has valid zero-boardings observations)
zero_rides_count = (df['rides'] == 0).sum()
print(f"True zero-count boarding observations: {zero_rides_count}")

Initial rows: 1,120,604
Duplicates removed: 0
Clean unique observations: 1,120,604
True zero-count boarding observations: 36


## Extract 2-Year Demand Window & Export Parquet
Extract observations from `2024-08-01` through `2026-07-31` for primary demand modeling[cite: 1].

In [4]:
# Extract recent 2-year demand modeling window
recent_df = df[(df['date'] >= '2024-08-01') & (df['date'] <= '2026-07-31')].copy()

# Save both complete historical and recent 2-year datasets to Parquet
full_parquet_path = os.path.join(PROCESSED_DATA_PATH, "clean_boardings_full.parquet")
recent_parquet_path = os.path.join(PROCESSED_DATA_PATH, "clean_boardings_2yr.parquet")

df.to_parquet(full_parquet_path, index=False)
recent_df.to_parquet(recent_parquet_path, index=False)

# Validation Summary
print("================ DATA VALIDATION SUMMARY ================")
print(f"Full Clean Dataset Range: {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}")
print(f"Full Clean Row Count    : {len(df):,}")
print(f"Unique Routes Count     : {df['route'].nunique()}")
print("---------------------------------------------------------")
print(f"2-Year Window Range     : {recent_df['date'].min().strftime('%Y-%m-%d')} to {recent_df['date'].max().strftime('%Y-%m-%d')}")
print(f"2-Year Window Row Count : {len(recent_df):,}")
print(f"Saved files to          : {PROCESSED_DATA_PATH}")
print("=========================================================")

================ DATA VALIDATION SUMMARY ================
Full Clean Dataset Range: 2001-01-01 to 2026-07-31
Full Clean Row Count    : 1,120,604
Unique Routes Count     : 188
---------------------------------------------------------
2-Year Window Range     : 2024-08-01 to 2026-07-31
2-Year Window Row Count : 83,997
Saved files to          : ..\data\processed
